# Imports

In [25]:
from delta import *
from pathlib import Path
from custom_builder import builder
from log import *

DB_SRC = "data"

BASE_DIR = Path("spark_project")
WAREHOUSE_DIR = BASE_DIR / "spark-warehouse"
METASTORE_DIR = BASE_DIR / "metastore_db"


spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')


from pyspark.sql import functions as F

import json
from pathlib import Path

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips_partitioned_by_county', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips_partitioned_by_date', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


26/09/12 14:54:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# Base taxi_trips queries

## Query 1

In [26]:
with log_step("query_1_taxi_trips") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False)

+-------------+---------+
|county       |row_count|
+-------------+---------+
|Manhattan    |2646947  |
|Queens       |273124   |
|Brooklyn     |25255    |
|Bronx        |6905     |
|Staten Island|72       |
+-------------+---------+



## Query 2

In [27]:
with log_step("query_2_taxi_trips") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

+----------+------------------+
|trip_date |avg_trip_seconds  |
+----------+------------------+
|2002-12-31|362.0             |
|2009-01-01|1657.0            |
|2023-12-31|609.5             |
|2024-01-01|982.1242917628471 |
|2024-01-02|1010.2056120557232|
|2024-01-03|989.8031883917306 |
|2024-01-04|962.5796946520375 |
|2024-01-05|920.6442417663364 |
|2024-01-06|872.8224700358335 |
|2024-01-07|832.2575692521801 |
|2024-01-08|934.5244039586145 |
|2024-01-09|901.7691333453953 |
|2024-01-10|912.9748939440193 |
|2024-01-11|983.5995733658388 |
|2024-01-12|985.2163661447329 |
|2024-01-13|902.8206437694496 |
|2024-01-14|860.5727554835362 |
|2024-01-15|891.8136878180496 |
|2024-01-16|993.3507861280373 |
|2024-01-17|978.2628555377353 |
+----------+------------------+
only showing top 20 rows


## Query 3

In [28]:
with log_step("query_3_taxi_trips") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

+-------------+------------------+
|county       |avg_fare_amount   |
+-------------+------------------+
|Bronx        |32.442825494658344|
|Brooklyn     |28.502018623943865|
|Manhattan    |14.711072251718443|
|Queens       |49.842337547941476|
|Staten Island|43.75208347042402 |
+-------------+------------------+



# Create partitioned taxi_trips

In [4]:

df = spark.read.format("delta").table("taxi_trips")

df = df.withColumn("trip_date", F.to_date("pu_datetime"))

df.write.format("delta") \
    .partitionBy("trip_date") \
    .mode("overwrite") \
    .saveAsTable("taxi_trips_partitioned_by_date")

26/09/12 14:22:50 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`default`.`taxi_trips_partitioned_by_date` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/12 14:22:50 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/09/12 14:22:50 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


In [5]:
df = spark.read.format("delta").table("taxi_trips")

new_df = spark.sql("""
        SELECT 
            pu_datetime, do_datetime, pu_location_id, do_location_id, fare_amount, county
        FROM default.taxi_trips t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
    """)

new_df.show(5, truncate=False)

new_df.write.format("delta") \
    .partitionBy("county") \
    .mode("overwrite") \
    .saveAsTable("taxi_trips_partitioned_by_county")

+-------------------+-------------------+--------------+--------------+-----------+---------+
|pu_datetime        |do_datetime        |pu_location_id|do_location_id|fare_amount|county   |
+-------------------+-------------------+--------------+--------------+-----------+---------+
|2024-01-24 15:17:12|2024-01-24 15:34:53|239           |246           |20.5       |Manhattan|
|2024-01-24 15:52:24|2024-01-24 16:01:39|234           |249           |10.7       |Manhattan|
|2024-01-24 15:08:55|2024-01-24 15:31:35|88            |211           |25.4       |Manhattan|
|2024-01-24 15:42:55|2024-01-24 15:51:35|211           |234           |9.3        |Manhattan|
|2024-01-24 15:52:23|2024-01-24 16:12:53|68            |144           |18.4       |Manhattan|
+-------------------+-------------------+--------------+--------------+-----------+---------+
only showing top 5 rows


26/09/12 14:22:55 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`default`.`taxi_trips_partitioned_by_county` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


## Storage statistics

In [ ]:
# table details of partitioned tables
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").show(truncate=False)

# calculate the number of rows in each partitioned table # TODO: Figure out why number of rows differ
spark.sql("SELECT COUNT(*) AS row_count FROM taxi_trips_partitioned_by_date").show(truncate=False)
spark.sql("SELECT COUNT(*) AS row_count FROM taxi_trips_partitioned_by_county").show(truncate=False)

# numFiles of table on disk
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").select("numFiles").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").select("numFiles").show(truncate=False)

# sizeInBytes of table on disk
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_date").select("sizeInBytes").show(truncate=False)
spark.sql("DESCRIBE DETAIL taxi_trips_partitioned_by_county").select("sizeInBytes").show(truncate=False)


+------+------------------------------------+----------------------------------------------------+-----------+-----------------------------------------------------------------------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name                                                |description|location                                                                                             |createdAt              |lastModified           |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+----------------------------------------------------+-----------+-----------------------------------------------------------------------------------------------------+--------------

# Drop partitioned tables

In [3]:
spark.sql("DROP TABLE IF EXISTS taxi_trips_partitioned_by_date")
spark.sql("DROP TABLE IF EXISTS taxi_trips_partitioned_by_county")

DataFrame[]

# Benchmark queries on partitioned tables

## Query 1

In [ ]:

with log_step("query_1_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips_partitioned_by_date t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False) 

+-------------+---------+
|county       |row_count|
+-------------+---------+
|Manhattan    |2646947  |
|Queens       |273124   |
|Brooklyn     |25255    |
|Bronx        |6905     |
|Staten Island|72       |
+-------------+---------+



In [ ]:
with log_step("query_1_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT
            m.county,
        COUNT(*) AS row_count
        FROM default.taxi_trips_partitioned_by_county t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY m.county
        ORDER BY row_count DESC;
    """)
result.show(truncate=False) 

+-------------+---------+
|county       |row_count|
+-------------+---------+
|Manhattan    |2646947  |
|Queens       |273124   |
|Brooklyn     |25255    |
|Bronx        |6905     |
|Staten Island|72       |
+-------------+---------+



## Query 2

In [ ]:
with log_step("query_2_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips_partitioned_by_date
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

+----------+------------------+
|trip_date |avg_trip_seconds  |
+----------+------------------+
|2002-12-31|362.0             |
|2009-01-01|1657.0            |
|2023-12-31|609.5             |
|2024-01-01|982.1242917628471 |
|2024-01-02|1010.2056120557232|
|2024-01-03|989.8031883917306 |
|2024-01-04|962.5796946520375 |
|2024-01-05|920.6442417663364 |
|2024-01-06|872.8224700358335 |
|2024-01-07|832.2575692521801 |
|2024-01-08|934.5244039586145 |
|2024-01-09|901.7691333453953 |
|2024-01-10|912.9748939440193 |
|2024-01-11|983.5995733658388 |
|2024-01-12|985.2163661447329 |
|2024-01-13|902.8206437694496 |
|2024-01-14|860.5727554835362 |
|2024-01-15|891.8136878180496 |
|2024-01-16|993.3507861280373 |
|2024-01-17|978.2628555377353 |
+----------+------------------+
only showing top 20 rows


In [17]:
with log_step("query_2_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT
            to_date(pu_datetime) AS trip_date,
            AVG(timestampdiff(SECOND, pu_datetime, do_datetime)) AS avg_trip_seconds
        FROM taxi_trips_partitioned_by_county
        WHERE do_datetime IS NOT NULL
        GROUP BY to_date(pu_datetime)
        ORDER BY trip_date;
    """)
result.show(truncate=False)

+----------+------------------+
|trip_date |avg_trip_seconds  |
+----------+------------------+
|2002-12-31|362.0             |
|2009-01-01|1657.0            |
|2023-12-31|609.5             |
|2024-01-01|981.5416444372759 |
|2024-01-02|1011.1988955422488|
|2024-01-03|990.177794302293  |
|2024-01-04|963.4131531936079 |
|2024-01-05|921.3129916660176 |
|2024-01-06|873.047903200786  |
|2024-01-07|832.14244263905   |
|2024-01-08|934.9881548403288 |
|2024-01-09|901.4051410203014 |
|2024-01-10|913.4565389612447 |
|2024-01-11|984.0158086512438 |
|2024-01-12|985.52928305175   |
|2024-01-13|903.5110325467315 |
|2024-01-14|860.3608145470013 |
|2024-01-15|892.0531980119232 |
|2024-01-16|993.6949889891619 |
|2024-01-17|976.9867527947595 |
+----------+------------------+
only showing top 20 rows


## Query 3

In [ ]:
with log_step("query_3_taxi_trips_partitioned_by_date") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips_partitioned_by_date t
        JOIN default.taxi_zone_lookup m
            ON t.pu_location_id = m.location_id
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

+-------------+------------------+
|county       |avg_fare_amount   |
+-------------+------------------+
|Bronx        |32.442825494658344|
|Brooklyn     |28.502018623943865|
|Manhattan    |14.711072251718443|
|Queens       |49.842337547941476|
|Staten Island|43.75208347042402 |
+-------------+------------------+



In [24]:
with log_step("query_3_taxi_trips_partitioned_by_county") as info:
    result = spark.sql("""
        SELECT 
            county,
            AVG(fare_amount) AS avg_fare_amount
        FROM default.taxi_trips_partitioned_by_county t
        GROUP BY county
        ORDER BY county;
    """)
result.show(truncate=False)

+-------------+------------------+
|county       |avg_fare_amount   |
+-------------+------------------+
|Bronx        |32.442825494658344|
|Brooklyn     |28.502018623943865|
|Manhattan    |14.711072251718443|
|Queens       |49.842337547941476|
|Staten Island|43.75208347042402 |
+-------------+------------------+

